In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")
print("Path to dataset files:", path)

In [ ]:
#imports
%pip install kagglehub catboost lightgbm tqdm -q
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(os.path.join(path, "Q3_data.csv"))


In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()
# you guys are repeating the same steps in Q1 file.

In [ ]:
# Task 1: Write your code here:
print(df.isnull().sum())

num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# note: if you are ever asking why is this, it showing this becuase I have already handled it by a diffrent function and run it again

In [ ]:
# Task 2: Write your code here:
df = df.drop_duplicates()


In [ ]:
# Task 3: Write your code here:
categorical_features = df.select_dtypes(include=['object']).columns
df = pd.get_dummies(df, columns=categorical_features, drop_first=True)

In [ ]:
# Task 4: Write your code here:
scaler = StandardScaler()
features_to_scale = df.drop('Target', axis=1).columns
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])


In [ ]:
# Task 5: Write your code here:
imbalance = df['Target'].value_counts(normalize=True)
print("the target distribution:\n", imbalance)

if imbalance.max() > 0.6:
    print("The dataset is imbalanced.")
else:
    print("The dataset is  balanced.")



In [ ]:
# Task 1: Write your code here:
X = df.drop('Target', axis=1)
y = df['Target']


In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

# Using F1-score becuase its imbalanced
is_imbalanced = y.value_counts(normalize=True).min() < 0.4

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0, random_seed=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    if is_imbalanced:
        score = f1_score(y_test, preds)
        metric_name = "F1-Score"
    else:
        score = accuracy_score(y_test, preds)
        metric_name = "Accuracy"

    scores.append(score)

print(f"Average {metric_name} across folds: {np.mean(scores):.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

importances = model.get_feature_importance()
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance_df['Feature'][:10], feature_importance_df['Importance'][:10])
plt.gca().invert_yaxis()
plt.title('Top 10 Feature Importances')
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_feature = feature_importance_df.iloc[0]['Feature']
print(f"The Golden Feature is: {golden_feature}")


In [ ]:
# Task Bonus: Write your code here:
# 1. first we create new golden_X
X_golden = X[[golden_feature]]

golden_scores = []

# 2. training with kfold again
for train_index, test_index in skf.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model_golden = CatBoostClassifier(iterations=500, verbose=0, random_seed=42)
    model_golden.fit(X_train, y_train)

    preds = model_golden.predict(X_test)
    # Acc becuase the Q saaid want acc
    golden_scores.append(accuracy_score(y_test, preds))

# 3. comparing
full_model_accuracy = np.mean(scores) if not is_imbalanced else accuracy_score(y, model.predict(X))
print(f"Full Model Performance: {full_model_accuracy:.4f}")
print(f"Golden Feature Only Performance: {np.mean(golden_scores):.4f}")
#btw, it's taking too long to train.
#also idk but i think we should have used the f1.